In [31]:
import os 


In [41]:
%pwd

'c:\\Users\\Administrator'

In [42]:
import os
os.chdir("c:/users/administrator/desktop/chicken dieses/chicken-dieses-classification-")

In [43]:
%pwd

'c:\\users\\administrator\\desktop\\chicken dieses\\chicken-dieses-classification-'

In [33]:
os.chdir("../")

In [44]:
%pwd

'c:\\users\\administrator\\desktop\\chicken dieses\\chicken-dieses-classification-'

In [45]:
#بس يخلص رح يروح ل 
#انتيتي 
"""
بدنا ننزل داتا من رابط ونخزنها بمسارات معينة مسجلة بملف الـ YAML. لو اعتمدنا على قواميس عادية ونصوص حرة، أي غلطة حرف صغيرة بتفركش الشغل. عملنا كلاس "كيان" مقفل (frozen=True) زي وصل استلام رسمي مطبوع ومحدد فيه خانات ثابتة: الرابط، ومكان الحفظ، ومكان فك الضغط.   


"""
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [46]:
#هذا رح يروح لل constant  
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories


In [47]:
#src/cnnClassifier/config/configuration.py
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])



    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [48]:
#هسا الكود الجاي ع المكونات 
import os
import urllib.request as request
import zipfile
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [49]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
    #دالة تنزيل الملف 
    def download_file(self):
        #طيب بشيك بالاول هل الملف موجود ولا لا 
        if not os.path.exists(self.config.local_data_file):
            #اذا مش موجود بروح بنزله من الرابط وبحفظه بالمكان الي حددته سلف
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")
        #داله فك الضغط 
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

            #ملف المكونات هو الورشه الي للتنفيذ الفعلي كل مرحله بالمشروع بكون الها مكون خاص فيها 
            #زي تنزيل الداتا مثلا ملف تجهيز الداتا ملف التدريب وهكذا 


In [61]:
#هسا بدنا نعمل ال pipline 
#هذا الكود هو كبسة التغشيل الفعلي لخط الانتاجب الكامل  
#بربط هسا كل الشغل فوق بهاي الخليه 
try:
    #صحيت مدير الاعدادات فتح ملف اليامل وانشا مجلد ال artifacts ع جهازي 
    config = ConfigurationManager()
   #المدير انشا مجلد ال 
   #artifacts\data_ingestation
    #وعبّى كل الروابط والمسارات داخل قالب الـ Entity وسلمنا إياه جاهز.
    data_ingestion_config = config.get_data_ingestion_config()
    #مسكت ال entity المعبيه وسلمتها بالايد لعامل التنزيل 
    data_ingestion = DataIngestion(config=data_ingestion_config)
    
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-11-06 23:31:08,384: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-11-06 23:31:08,387: INFO: common: yaml file: params.yaml loaded successfully]
[2026-11-06 23:31:08,389: INFO: common: created directory at: artifacts]
[2026-11-06 23:31:08,391: INFO: common: created directory at: artifacts/data_ingestion]
[2026-11-06 23:31:11,754: INFO: 1291316891: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 11616915
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "adf745abc03891fe493c3be264ec012691fe3fa21d861f35a27edbe6d86a76b1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: FCAA:32CB9B:570D9:5ABE6:6AEE390D
x-github-edge-region: fra
Accept-Ranges: bytes
Date: Fri, 06 Nov 2026 20:31:10 GMT
Via: 1.1 varnish
X-Served-By: cach